In [0]:
!pip install torch torchvision opencv-python ultralytics

In [0]:

# 1. IMPORT LIBRARIES & SET INLINE PLOTTING

%matplotlib inline
from ultralytics import YOLO
import cv2
import matplotlib.pyplot as plt
from PIL import Image
import io


# 2. DEFINE PATHS TO MODEL WEIGHTS AND IMAGE FRAME

model_weights_path = "/XXX/yolo12x.pt" # Point to your downloaded YOLO model weights somewhere 
image_path = "/xxx/Eem/Decoded Frames/Eem_ch04_0619_060343_235956/video1/0000001.jpg"  # Point to the initial picture in the decoded batch


# 3. INITIALIZE THE YOLOv12 MODEL

model = YOLO(model_weights_path)


# 4. RUN PREDICTION ON THE IMAGE

results = model(image_path)


# 5. PRINT OUT THE PREDICTION DETAILS

print("Detected Objects:")
for result in results:
    if result.boxes is not None:
        for box in result.boxes:
            # Extract bounding box coordinates, confidence, and class index
            coords = box.xyxy[0].tolist()  # [x1, y1, x2, y2]
            conf = box.conf[0].item()
            cls = int(box.cls[0].item())
            class_name = model.names[cls] if model.names and cls in model.names else str(cls)
            print(f"Class: {class_name}, Confidence: {conf:.2f}, BBox: {coords}")


# 6. VISUALIZE THE BOUNDING BOXES ON THE ORIGINAL IMAGE

# Load the original image using OpenCV (BGR format)
image = cv2.imread(image_path)
if image is None:
    raise ValueError(f"Could not load the image from {image_path}")

# Loop through each detected bounding box and draw on the image
for result in results:
    if result.boxes is not None:
        for box in result.boxes:
            # Convert coordinates to integers
            x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
            conf = box.conf[0].item()
            cls = int(box.cls[0].item())
            class_name = model.names[cls] if model.names and cls in model.names else str(cls)
            label = f"{class_name}: {conf:.2f}"
            
            # Draw the bounding box (green rectangle)
            cv2.rectangle(image, (x1, y1), (x2, y2), color=(0, 255, 0), thickness=2)
            # Draw a label background for better visibility
            (text_width, text_height), baseline = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
            cv2.rectangle(image, (x1, y1 - text_height - baseline), (x1 + text_width, y1), (0, 255, 0), -1)
            # Put the label text above the bounding box
            cv2.putText(image, label, (x1, y1 - baseline), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), thickness=1)

# Convert image from BGR to RGB for proper display with matplotlib and PIL
image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)


# 7A. DISPLAY THE PLOT USING MATPLOTLIB

plt.figure(figsize=(10, 10))
plt.imshow(image_rgb)
plt.axis("on")
plt.title("Detection Results on Original Image")
plt.show()


# 7B. IF THE MATPLOTLIB PLOT DOESN'T APPEAR, USE THE DATBRICKS display() FUNCTION

# Convert the NumPy image array to a PIL image and display it:
pil_img = Image.fromarray(image_rgb)
display(pil_img)